# Customer Churn Prediction: Profit-Based Evaluation of ML Models
## Step 3: Exploratory Data Analysis (EDA) & Visualization

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
CHURN_PALETTE = {1: '#E74C3C', 0: '#2E86AB'}  # red = churn, blue = no churn

df = pd.read_csv('churn_cleaned.csv')
print(f'Cleaned data loaded: {df.shape[0]} rows × {df.shape[1]} columns ✓')

### 3.1 — Churn Rate by Contract Type

In [ ]:
contract_churn = df.groupby('Contract')['Churn'].mean() * 100
contract_count = df.groupby('Contract')['Churn'].count()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(contract_churn.index, contract_churn.values,
              color=['#E74C3C', '#F39C12', '#2E86AB'], edgecolor='white', width=0.5)

for bar, val, cnt in zip(bars, contract_churn.values, contract_count.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{val:.1f}%\n(n={cnt:,})', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Contract Type', fontsize=12)
ax.set_ylabel('Churn Rate (%)', fontsize=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(0, 60)
plt.tight_layout()
plt.savefig('plot_04_churn_by_contract.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInsight: Month-to-month customers churn at a dramatically higher rate than')
print('long-term contract customers. Contract type is likely a strong predictor.')

### 3.2 — Tenure vs Churn (How long before customers leave?)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram by churn group
for churn_val, label, color in [(0, 'No Churn', '#2E86AB'), (1, 'Churned', '#E74C3C')]:
    subset = df[df['Churn'] == churn_val]['tenure']
    axes[0].hist(subset, bins=30, alpha=0.6, label=label, color=color, edgecolor='white')

axes[0].set_title('Tenure Distribution by Churn', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Number of Customers')
axes[0].legend()

# Boxplot
df_plot = df.copy()
df_plot['Churn Label'] = df_plot['Churn'].map({1: 'Churned', 0: 'No Churn'})
sns.boxplot(data=df_plot, x='Churn Label', y='tenure', ax=axes[1],
            palette={'Churned': '#E74C3C', 'No Churn': '#2E86AB'})
axes[1].set_title('Tenure Boxplot by Churn', fontweight='bold', fontsize=13)
axes[1].set_xlabel('')
axes[1].set_ylabel('Tenure (months)')

plt.suptitle('Customer Tenure vs Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_05_tenure_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Avg tenure — Churned: {df[df['Churn']==1]['tenure'].mean():.1f} months")
print(f"Avg tenure — Retained: {df[df['Churn']==0]['tenure'].mean():.1f} months")
print('\nInsight: Churned customers tend to leave early (low tenure).')
print('Customers who stay past ~24 months are much less likely to churn.')

### 3.3 — Monthly Charges vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Density plot
for churn_val, label, color in [(0, 'No Churn', '#2E86AB'), (1, 'Churned', '#E74C3C')]:
    subset = df[df['Churn'] == churn_val]['MonthlyCharges']
    subset.plot.kde(ax=axes[0], label=label, color=color, linewidth=2.5)

axes[0].set_title('Monthly Charges — Density by Churn', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Boxplot
sns.boxplot(data=df_plot, x='Churn Label', y='MonthlyCharges', ax=axes[1],
            palette={'Churned': '#E74C3C', 'No Churn': '#2E86AB'})
axes[1].set_title('Monthly Charges Boxplot by Churn', fontweight='bold', fontsize=13)
axes[1].set_xlabel('')
axes[1].set_ylabel('Monthly Charges ($)')

plt.suptitle('Monthly Charges vs Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_06_monthly_charges_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Avg monthly charge — Churned: ${df[df['Churn']==1]['MonthlyCharges'].mean():.2f}")
print(f"Avg monthly charge — Retained: ${df[df['Churn']==0]['MonthlyCharges'].mean():.2f}")
print('\nInsight: Churned customers tend to pay higher monthly charges.')
print('High-value customers are at greater risk — critical for profit-based evaluation.')

### 3.4 — Churn Rate Across Key Categorical Features

In [ ]:
cat_features = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'InternetService', 'PaymentMethod'
]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    churn_rate = df.groupby(col)['Churn'].mean() * 100
    colors = ['#E74C3C' if v > 26 else '#2E86AB' for v in churn_rate.values]
    bars = axes[i].bar(churn_rate.index.astype(str), churn_rate.values,
                       color=colors, edgecolor='white')
    for bar, val in zip(bars, churn_rate.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    axes[i].set_title(col, fontweight='bold', fontsize=11)
    axes[i].set_ylabel('Churn Rate (%)')
    axes[i].set_ylim(0, max(churn_rate.values) + 10)
    axes[i].tick_params(axis='x', rotation=15)

# Hide the last empty subplot
axes[-1].set_visible(False)

plt.suptitle('Churn Rate by Categorical Features', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plot_07_categorical_churn_rates.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight: Senior citizens, Fibre optic users, and Electronic check payment')
print('customers show notably higher churn rates.')

### 3.5 — Correlation Heatmap (Numeric Features)

In [ ]:
num_df = df[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # show full matrix
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r',
            ax=ax, linewidths=0.5, vmin=-1, vmax=1,
            annot_kws={'size': 12, 'weight': 'bold'})
ax.set_title('Correlation Matrix — Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_08_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insights:')
print(f"  tenure       vs Churn: {corr.loc['tenure','Churn']:.2f}  (negative — longer tenure = less churn)")
print(f"  MonthlyCharges vs Churn: {corr.loc['MonthlyCharges','Churn']:.2f}  (positive — higher charges = more churn)")
print(f"  TotalCharges vs Churn: {corr.loc['TotalCharges','Churn']:.2f}")
print(f"  tenure vs TotalCharges: {corr.loc['tenure','TotalCharges']:.2f}  (high — multicollinearity to watch)")

### 3.6 — Profit Proxy: Monthly Charges × Tenure

In [ ]:
# This is the key insight linking EDA to your profit-based evaluation
# Customers who churn early AND pay more = highest financial loss

fig, ax = plt.subplots(figsize=(9, 6))

for churn_val, label, color, alpha in [(0, 'No Churn', '#2E86AB', 0.3), (1, 'Churned', '#E74C3C', 0.5)]:
    subset = df[df['Churn'] == churn_val]
    ax.scatter(subset['tenure'], subset['MonthlyCharges'],
               c=color, label=label, alpha=alpha, s=15)

ax.set_title('Tenure vs Monthly Charges — Coloured by Churn\n(Top-left = Early high-paying churners = biggest loss)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Tenure (months)', fontsize=12)
ax.set_ylabel('Monthly Charges ($)', fontsize=12)
ax.legend(fontsize=11)

# Annotate the high-risk zone
ax.axvspan(0, 12, alpha=0.07, color='red')
ax.text(6, 115, 'High-risk\nzone', ha='center', color='darkred', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('plot_09_tenure_charges_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight: The red high-risk zone (low tenure, high monthly charges) represents')
print('customers whose early departure causes the greatest financial loss.')
print('This motivates the profit-based evaluation in Steps 6 & 7.')

In [ ]:
print('='*55)
print('           EDA SUMMARY — KEY FINDINGS')
print('='*55)
print('  1. Churn rate is 26.5% — class imbalance present')
print('  2. Month-to-month contracts churn most (~42%)')
print('  3. Churned customers have much lower avg tenure')
print('  4. Churned customers pay higher monthly charges')
print('  5. Senior citizens and Fibre optic users = higher risk')
print('  6. Electronic check payment = highest churn payment method')
print('  7. High-risk zone: low tenure + high charges = max loss')
print('='*55)
print('Step 3 Complete ✓  →  Proceed to Step 4: Preprocessing & Modelling')